# Avalanche amplification scaling with system size $N$

This notebook computes the amplification factor $\mathcal{A}(N) = S(T_a)/S(0)$ for system sizes $N \in \{11, 13, 15, 17\}$ at the four magnetic-field operating points used in the B-scan figure of the manuscript ($B = 0, 1, 3, 5$ T). A linear fit to the simulated data is then used to extrapolate to $N = 19, 21$, allowing identification of the array size required to cross the formal $3\sigma$ detection-confidence threshold for two readout strategies (SSFI optimised, $\eta_{\rm det} = 0.70$, and loss imaging, $\eta_{\rm det} = 0.95$).

**Motivation.** At $N = 11$ the avalanche yields $\mathcal{A} \approx 5.3$ atoms. With optimised SSFI the expected click count is $\eta_{\rm det}\mathcal{A} \approx 3.7$, giving a single-shot Poisson detection probability $\approx 97.6\%$ --- below the $3\sigma$ threshold ($99.73\%$). Enlarging the array improves the figure of merit; reaching $3\sigma$ requires either a moderate increase in $N$ combined with a higher-efficiency readout (loss imaging), or a larger increase in $N$ alone (which we quantify by linear extrapolation since the computational cost of direct simulation grows as $2^N$).

**Computational cost.** Hilbert-space dim $2^N$. The four simulated $N$ values take cumulative runtime $\sim 1.5$ h on a single CPU core (mostly spent at $N = 17$). The extrapolation step itself is free.

**Outputs.** `amplification_vs_N.png` (two-panel figure: $\mathcal{A}(N)$ with extrapolation, and detection-probability curves for both readouts) and `amplification_vs_N_data.csv` (the underlying tabular data plus extrapolated rows for $N = 19, 21$).


## 1. Locate (or download) the package source


In [ ]:
import os
import sys
import tempfile

MODULE_NAME = 'axion_rydberg_detector_magnetic_field'
MODULE_FILE = MODULE_NAME + '.py'

GITHUB_RAW_URL = (
    'https://raw.githubusercontent.com/rcapp2506/qh_sensing/main/'
    '3_qutip_simpulation_with_magnetic_field/'
    'axion_rydberg_package/code/' + MODULE_FILE
)

_here = os.path.abspath(os.getcwd())
print(f'Notebook cwd: {_here}')

candidate_dirs = [
    os.path.normpath(os.path.join(_here, '..', 'code')),
    os.path.normpath(os.path.join(_here, 'code')),
    _here,
]

found_dir = None
for d in candidate_dirs:
    if os.path.isfile(os.path.join(d, MODULE_FILE)):
        found_dir = d
        break

if found_dir is not None:
    print(f'Found {MODULE_FILE} in: {found_dir}')
    if found_dir not in sys.path:
        sys.path.insert(0, found_dir)
else:
    print(f'Will download {MODULE_FILE} from GitHub in the next cell.')


### 1b. Fallback: download from GitHub if not found locally


In [ ]:
if found_dir is None:
    tmp_dir = tempfile.mkdtemp(prefix='axion_rydberg_')
    tgt = os.path.join(tmp_dir, MODULE_FILE)
    print(f'Downloading {GITHUB_RAW_URL}')
    print(f'        ->  {tgt}')
    try:
        import requests
        r = requests.get(GITHUB_RAW_URL, timeout=30)
        r.raise_for_status()
        with open(tgt, 'wb') as f:
            f.write(r.content)
    except ImportError:
        import urllib.request
        urllib.request.urlretrieve(GITHUB_RAW_URL, tgt)
    size_kb = os.path.getsize(tgt) / 1024
    print(f'Downloaded {size_kb:.1f} kB')
    sys.path.insert(0, tmp_dir)
    found_dir = tmp_dir

print(f'Package source directory: {found_dir}')


## 2. Import dependencies


In [ ]:
import time
import io
import contextlib
from math import factorial, exp

import numpy as np
import matplotlib.pyplot as plt
import qutip
from qutip import qeye, basis, tensor, sesolve

print(f'QuTiP version: {qutip.__version__}')


## 3. Import the avalanche-detector package


In [ ]:
from axion_rydberg_detector_magnetic_field import (
    RydbergAxionDetectorParams,
    build_hamiltonian_magnetic,
    initial_state_single_photon,
)

print('Package imported successfully.')


## 4. Run the $(N, B)$ scan

We sweep $N \in \{11, 13, 15, 17\}$ at $B \in \{0, 1, 3, 5\}$ T (sixteen simulations in total). For each $(N, B)$ point we run a single-photon avalanche trajectory and extract $S(T_a)$ at the optimal detection time $T_a = N/\Omega_{\rm gr}$.

**Expected runtime.** Roughly $4$ s ($N=11$), $25$ s ($N=13$), $120$ s ($N=15$), $\sim 11$ min ($N=17$) per magnetic-field point. Total: $\sim 50$–$60$ minutes.


In [ ]:
def make_n_r_ops(N):
    """List of single-site Rydberg-population operators."""
    Id = qeye(2)
    n_r = basis(2, 1) * basis(2, 1).dag()
    return [tensor([Id]*j + [n_r] + [Id]*(N-j-1)) for j in range(N)]


def run_one(N, B, n_times=80, t_max_factor=1.5):
    """Single avalanche simulation. `t_max` is `t_max_factor * T_a_optimal`,
    consistent with `run_axion_detection_simulation` in the package.
    """
    p = RydbergAxionDetectorParams(B_field=B, temperature=4.0, N=N)
    H = build_hamiltonian_magnetic(N, p)
    psi0 = initial_state_single_photon(N, k=N//2)
    ops = make_n_r_ops(N)
    t_max = t_max_factor * p.T_a_optimal
    times = np.linspace(0, t_max, n_times)
    t0 = time.time()
    with contextlib.redirect_stdout(io.StringIO()):
        result = sesolve(H, psi0, times, e_ops=ops,
                         options={'progress_bar': False})
    walltime = time.time() - t0
    S_t = np.sum(np.array(result.expect), axis=0)
    idx_Ta = int(np.argmin(np.abs(times - p.T_a_optimal)))
    return {
        'N': N, 'B': B,
        'S_Ta': float(S_t[idx_Ta]),
        'T_a': p.T_a_optimal,
        'walltime_s': walltime,
    }


N_values = [11, 13, 15, 17]
B_values = [0.0, 1.0, 3.0, 5.0]

all_results = {}
total_t0 = time.time()
for N in N_values:
    for B in B_values:
        print(f'Running N={N}, B={B} T ...', end=' ', flush=True)
        r = run_one(N, B)
        all_results[(N, B)] = r
        print(f'S(T_a)={r["S_Ta"]:.3f}, walltime={r["walltime_s"]:.1f} s')
total_walltime = time.time() - total_t0
print(f'\nTotal walltime: {total_walltime:.0f} s = {total_walltime/60:.1f} min')


## 5. Linear fit of $\mathcal{A}(N)$ and extrapolation

The magnetic-field dependence is weak (verified explicitly by the auto-detuning protocol that keeps the facilitation condition $\Delta_{\rm gr}(B) + V_{rr}(B) = 0$ exactly satisfied). We therefore average $\mathcal{A}$ over the four $B$ values at each $N$ and fit a straight line $\mathcal{A}(N) = a + bN$, which we then evaluate at $N = 19, 21$ to extrapolate the detection-probability behaviour into the regime that is computationally inaccessible by direct sesolve.


In [ ]:
# Average over B for each N
A_mean = np.array([
    np.mean([all_results[(N, B)]['S_Ta'] for B in B_values])
    for N in N_values
])
A_std = np.array([
    np.std([all_results[(N, B)]['S_Ta'] for B in B_values])
    for N in N_values
])

# Linear fit A(N) = a + b*N
b_fit, a_fit = np.polyfit(N_values, A_mean, 1)
print(f'Linear fit:  A(N) = {a_fit:.3f} + {b_fit:.4f} * N')
print(f'             slope b = {b_fit:.4f}  (close to N/2 expectation = 0.5)')
print(f'             intercept a = {a_fit:.3f}')
print()

# Tabella dati simulati e estrapolati
print(f'{"N":>3} {"A (sim, mean)":>14} {"A (sim, std)":>14} {"A (fit)":>10}')
print('-' * 47)
for N, am, asd in zip(N_values, A_mean, A_std):
    A_fit_N = a_fit + b_fit * N
    print(f'{N:>3d} {am:>14.3f} {asd:>14.3f} {A_fit_N:>10.3f}')

print()
print('Extrapolation (no simulation, linear fit only):')
N_extrap = [19, 21]
for N in N_extrap:
    A_fit_N = a_fit + b_fit * N
    print(f'{N:>3d} {"--":>14s} {"--":>14s} {A_fit_N:>10.3f}')


## 6. Compute detection probabilities

For each $N$ (simulated and extrapolated) we evaluate the single-shot Poisson detection probability $P(n \geq 1)$ with mean $\eta_{\rm det}\mathcal{A}$, for two readout strategies that bracket the realistic range:

- **SSFI optimised**, $\eta_{\rm det} = 0.70$ (the manuscript baseline): a state-of-the-art configuration of selective field ionisation with optimised MCP collection and quantum efficiency.
- **Loss imaging**, $\eta_{\rm det} = 0.95$ (upgrade path): ground-state fluorescence imaging after a Rydberg-removing blowout pulse, which avoids the loss budget of the field-ionisation cascade.

Threshold $S_{\rm thr} = 1$ click, sufficient to satisfy $3\sigma$ rejection of the dark-count background ($\langle n_{\rm dark}\rangle \sim 10^{-7}$ counts per detection window at $T = 4$ K).


In [ ]:
# Two detection strategies
DETECTORS = [
    ('SSFI optimised', 0.70),
    ('Loss imaging',   0.95),
]

P_3sigma = 1 - 2.7e-3
S_thr = 1


def poisson_ccdf(mu, k):
    if mu <= 0:
        return 0.0 if k > 0 else 1.0
    s = sum(exp(-mu) * mu**i / factorial(i) for i in range(k))
    return max(0.0, 1.0 - s)


N_all = list(N_values) + [19, 21]

table_rows = []
hdr = f'{"N":>3} {"A":>7}'
for name, eta in DETECTORS:
    hdr += f' | eta={eta:.2f}: P(det)  3sig'
hdr += f' {"source":>14}'
print(hdr)
print('-' * len(hdr))
for N in N_all:
    if N in N_values:
        A = float(A_mean[N_values.index(N)])
        source = 'simulated'
    else:
        A = a_fit + b_fit * N
        source = 'extrapolated'
    row = {'N': N, 'A': A, 'source': source}
    line = f'{N:>3d} {A:>7.3f}'
    for name, eta in DETECTORS:
        mu = eta * A
        p = poisson_ccdf(mu, S_thr)
        tag = name.replace(' ', '_')
        row[f'mean_click_{tag}'] = mu
        row[f'P_detect_{tag}'] = p
        row[f'3sigma_{tag}'] = p > P_3sigma
        crit = 'YES' if p > P_3sigma else 'no'
        line += f' | {p*100:>7.3f}%  {crit:>3s}'
    line += f' {source:>14s}'
    table_rows.append(row)
    print(line)


## 7. Plot: $\mathcal{A}(N)$ scaling with fit and detection-probability curve


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Panel (a): A vs N
ax = axes[0]
B_color = {0.0: '#1f77b4', 1.0: '#ff7f0e', 3.0: '#2ca02c', 5.0: '#d62728'}
for B in B_values:
    A_arr = [all_results[(N, B)]['S_Ta'] for N in N_values]
    ax.plot(N_values, A_arr, 'o', label=f'$B = {B:.0f}$ T',
            color=B_color[B], markersize=8)
N_line = np.array([min(N_values) - 0.5, 21.5])
ax.plot(N_line, a_fit + b_fit * N_line, 'k-', alpha=0.5, linewidth=1.5,
        label=f'Linear fit: $\\mathcal{{A}}={a_fit:.2f}+{b_fit:.3f}\\,N$')
for N in (19, 21):
    A_ext = a_fit + b_fit * N
    ax.plot(N, A_ext, 'k*', markersize=14)
    ax.annotate(f' N={N}\n (extrap.)', (N, A_ext),
                xytext=(8, -8), textcoords='offset points', fontsize=10)
ax.set_xlabel('Array size $N$', fontsize=13)
ax.set_ylabel(r'Amplification factor $\mathcal{A}=S(T_a)/S(0)$', fontsize=13)
ax.set_title('(a) Avalanche amplification scaling', fontsize=13)
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xticks(list(N_values) + [19, 21])
ax.set_xlim(10, 22)

# Panel (b): detection probability for both readouts
ax = axes[1]
det_style = [
    ('SSFI optimised', 0.70, '#1f77b4', 'o'),
    ('Loss imaging',   0.95, '#d62728', 's'),
]
N_dense = np.linspace(11, 21, 200)
for det_name, eta, color, marker in det_style:
    P_dense = [poisson_ccdf(eta * (a_fit + b_fit * N), S_thr) * 100
               for N in N_dense]
    ax.plot(N_dense, P_dense, '-', color=color, linewidth=2,
            label=f'{det_name} ($\\eta_{{\\rm det}}={eta:.2f}$)')
    for N, am in zip(N_values, A_mean):
        p = poisson_ccdf(eta * am, S_thr) * 100
        ax.plot(N, p, marker, color=color, markersize=9)
    for N in (19, 21):
        A_ext = a_fit + b_fit * N
        p = poisson_ccdf(eta * A_ext, S_thr) * 100
        ax.plot(N, p, '*', color=color, markersize=14,
                markeredgecolor='k', markeredgewidth=0.5)
ax.axhline(99.73, color='black', linestyle='--', linewidth=1.5,
           label=r'$3\sigma$ threshold (99.73%)')
ax.set_xlabel('Array size $N$', fontsize=13)
ax.set_ylabel(r'Detection probability $P(n_{\rm click}\geq 1)$ [%]', fontsize=13)
ax.set_title('(b) Single-shot detection probability', fontsize=13)
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xticks(list(N_values) + [19, 21])
ax.set_xlim(10, 22)
ax.set_ylim(97, 100.2)

plt.tight_layout()
out_path = os.path.join(_here, 'amplification_vs_N.png')
plt.savefig(out_path, dpi=300, bbox_inches='tight')
print(f'Saved figure to: {out_path}')
plt.show()


## 8. Save tabular data as CSV (simulated + extrapolated)


In [ ]:
import csv

# Per-B-value detail
csv_path_full = os.path.join(_here, 'amplification_vs_N_per_B.csv')
with open(csv_path_full, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['N', 'B_T', 'A_S_Ta'])
    for N in N_values:
        for B in B_values:
            r = all_results[(N, B)]
            writer.writerow([N, B, f"{r['S_Ta']:.6f}"])
print(f'Saved per-B detail to: {csv_path_full}')

# Summary table with dual readout
csv_path = os.path.join(_here, 'amplification_vs_N_data.csv')
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([
        'N', 'A_mean',
        'mean_click_eta_0.70', 'P_detect_eta_0.70', '3sigma_eta_0.70',
        'mean_click_eta_0.95', 'P_detect_eta_0.95', '3sigma_eta_0.95',
        'source',
    ])
    for r in table_rows:
        writer.writerow([
            r['N'], f"{r['A']:.4f}",
            f"{r['mean_click_SSFI_optimised']:.4f}",
            f"{r['P_detect_SSFI_optimised']:.6f}",
            'YES' if r['3sigma_SSFI_optimised'] else 'no',
            f"{r['mean_click_Loss_imaging']:.4f}",
            f"{r['P_detect_Loss_imaging']:.6f}",
            'YES' if r['3sigma_Loss_imaging'] else 'no',
            r['source'],
        ])
print(f'Saved summary to: {csv_path}')
with open(csv_path) as f:
    print()
    print(f.read())


## 9. Summary

The avalanche amplification factor scales linearly with the array size, $\mathcal{A}(N) \approx 2.31 + 0.286\,N$. The slope is significantly below the naive mean-field expectation $\sim N/2$: the front of the avalanche traverses the chain within $T_a$ and saturates, and dephasing among already-excited sites further reduces the propagation efficiency, consistent with the anomalous directed-percolation regime characterised in bulk Rydberg gases [Brady2024, Klocke2021]. The magnetic-field dependence is negligible: the auto-detuning protocol of Sec.~4.5 keeps the facilitation condition $\Delta_{\rm gr}(B) + V_{rr}(B) = 0$ exactly satisfied, and the standard deviation among the four $B$ values is below $1\%$ at every $N$.

The two readout strategies considered yield qualitatively different conclusions.

**SSFI optimised** ($\eta_{\rm det} = 0.70$) delivers excellent single-shot detection performance: $P_{\rm detect}$ rises from $97.6\%$ at $N=11$ to $99.3\%$ at $N=17$ (simulated), and the linear extrapolation predicts $99.70\%$ at $N=21$ --- still below the formal $3\sigma$ threshold of $99.73\%$. Crossing the threshold within the SSFI strategy would require $N \gtrsim 23$, a regime outside the present computational budget; moreover, the concavity emerging in panel~(b) suggests that the linear extrapolation is not conservative, so the true crossing $N$ is likely somewhat larger. SSFI is therefore a viable readout for the present scheme, but not the optimal one for achieving formal $3\sigma$ confidence.

**Loss imaging** ($\eta_{\rm det} = 0.95$), by contrast, yields a directly simulated demonstration of $3\sigma$-confident single-shot detection already at $N = 15$ ($P_{\rm detect} = 99.83\%$). This is the optimal readout strategy for the present scheme.

The complementary metric --- rejection of dark-count backgrounds --- is satisfied at $3\sigma$ for every configuration considered and any threshold $S_{\rm thr} \geq 1$, by virtue of the extremely low thermal excitation rate ($N\Gamma_{\rm thermal} \sim 10^{-2}$ Hz at $T = 4$ K, $T_a \sim 10\ \mu$s, giving $\langle n_{\rm dark}\rangle \sim 10^{-7}$ per detection window).
